Assignment 15

task 1

Create a CSV file named purchase_data.csv with columns: customer_id, age_group (18-25, 26-35, 36-50), and purchase_value. Enter at least 15 sample records with 5 customers in each age group.

In [1]:
import pandas as pd

# 15 sample records: 5 customers in each of the 3 age groups
purchase_data = {
    "customer_id": list(range(1, 16)),
    "age_group": ["18-25"]*5 + ["26-35"]*5 + ["36-50"]*5,
    "purchase_value": [1317, 1398, 1087, 1071, 1256,      # 18-25
                        1860, 1664, 1432, 1490, 1596,      # 26-35
                        2044, 1903, 1851, 1871, 1930]       # 36-50
}

purchase_df = pd.DataFrame(purchase_data)

# Save to a CSV file
purchase_df.to_csv("purchase_data.csv", index=False)

print("purchase_data.csv created with", len(purchase_df), "records\n")
print(purchase_df)


purchase_data.csv created with 15 records

    customer_id age_group  purchase_value
0             1     18-25            1317
1             2     18-25            1398
2             3     18-25            1087
3             4     18-25            1071
4             5     18-25            1256
5             6     26-35            1860
6             7     26-35            1664
7             8     26-35            1432
8             9     26-35            1490
9            10     26-35            1596
10           11     36-50            2044
11           12     36-50            1903
12           13     36-50            1851
13           14     36-50            1871
14           15     36-50            1930


**Explanation:** We build the 15 required records (5 per age group) as a pandas DataFrame and save it directly to `purchase_data.csv` using `to_csv()`. This file is then read back in Task 2, just as it would be if it had been created manually in Excel or Google Sheets.

task 2

Write a Python script using pandas and scipy.stats that reads purchase_data.csv and performs a one-way ANOVA to compare the average purchase_value across the three age groups. Print the F-statistic and p-value.

In [2]:
import pandas as pd
from scipy import stats

# Read the CSV file created in Task 1
df = pd.read_csv("purchase_data.csv")

# Split purchase_value into separate arrays, one per age group
group_18_25 = df[df["age_group"] == "18-25"]["purchase_value"]
group_26_35 = df[df["age_group"] == "26-35"]["purchase_value"]
group_36_50 = df[df["age_group"] == "36-50"]["purchase_value"]

print("Mean purchase value by age group:")
print(df.groupby("age_group")["purchase_value"].mean())

# Perform one-way ANOVA
f_stat, p_value = stats.f_oneway(group_18_25, group_26_35, group_36_50)

print(f"\nF-statistic = {round(f_stat, 4)}")
print(f"p-value = {round(p_value, 6)}")


Mean purchase value by age group:
age_group
18-25    1225.8
26-35    1608.4
36-50    1919.8
Name: purchase_value, dtype: float64

F-statistic = 33.4527
p-value = 1.2e-05


**Explanation:** `scipy.stats.f_oneway()` performs a one-way ANOVA, comparing the means of `purchase_value` across the three independent age groups at once (18-25, 26-35, 36-50), rather than needing separate pairwise t-tests. The F-statistic is the ratio of between-group variance to within-group variance (calculated explicitly in Task 4) — a larger F-statistic suggests the group means differ more than would be expected from random variation alone.

task 3

Based on your ANOVA results, interpret whether there is a statistically significant difference in average purchase_value between the age groups. Write your answer in 2-3 lines.

In [3]:
alpha = 0.05

if p_value < alpha:
    interpretation = (
        f"Since the p-value ({round(p_value, 6)}) is less than alpha (0.05), we reject the null "
        "hypothesis. There IS a statistically significant difference in average purchase_value "
        "between the three age groups - purchase behavior genuinely varies by age group, it isn't "
        "just random sample variation."
    )
else:
    interpretation = (
        f"Since the p-value ({round(p_value, 6)}) is greater than alpha (0.05), we fail to reject "
        "the null hypothesis. There is NOT enough evidence of a statistically significant "
        "difference in average purchase_value between the three age groups."
    )

print(interpretation)


Since the p-value (1.2e-05) is less than alpha (0.05), we reject the null hypothesis. There IS a statistically significant difference in average purchase_value between the three age groups - purchase behavior genuinely varies by age group, it isn't just random sample variation.


**Explanation:** The interpretation directly follows from comparing the p-value to alpha = 0.05, the standard significance threshold. With the sample data used here, the F-statistic is large and the p-value is well below 0.05, so we conclude there is a statistically significant difference in average purchase value across the 18-25, 26-35, and 36-50 age groups — older age groups in this sample tend to spend noticeably more per purchase.

task 4

Calculate and print the between-group variance and within-group variance for the purchase_value data using numpy or pandas. Explain briefly what each value represents in this context.

**Hint:** Between-group variance measures differences between age group means; within-group variance measures variation inside each age group.

In [4]:
import numpy as np

overall_mean = df["purchase_value"].mean()
groups = [group_18_25, group_26_35, group_36_50]
group_names = ["18-25", "26-35", "36-50"]

# Between-group variance (based on Sum of Squares Between, SSB)
ss_between = sum(len(g) * (g.mean() - overall_mean) ** 2 for g in groups)
df_between = len(groups) - 1
variance_between = ss_between / df_between

# Within-group variance (based on Sum of Squares Within, SSW)
ss_within = sum(((g - g.mean()) ** 2).sum() for g in groups)
df_within = len(df) - len(groups)
variance_within = ss_within / df_within

print(f"Overall mean purchase value = {round(overall_mean, 2)}")
print(f"\nSum of Squares Between (SSB) = {round(ss_between, 2)}")
print(f"Between-group variance = SSB / {df_between} = {round(variance_between, 2)}")

print(f"\nSum of Squares Within (SSW) = {round(ss_within, 2)}")
print(f"Within-group variance = SSW / {df_within} = {round(variance_within, 2)}")

# Sanity check: this ratio should match the F-statistic from Task 2
f_check = variance_between / variance_within
print(f"\nBetween-group variance / Within-group variance = {round(f_check, 4)} (should match the F-statistic from Task 2)")


Overall mean purchase value = 1584.67

Sum of Squares Between (SSB) = 1208314.53
Between-group variance = SSB / 2 = 604157.27

Sum of Squares Within (SSW) = 216720.8
Within-group variance = SSW / 12 = 18060.07

Between-group variance / Within-group variance = 33.4527 (should match the F-statistic from Task 2)


**Explanation:** **Between-group variance** measures how much the age groups' *means* differ from the overall mean — a high value means the age groups have very different average purchase values. **Within-group variance** measures how much individual purchase values differ from their *own* group's mean — a low value means customers within the same age group behave fairly consistently. ANOVA's F-statistic is exactly the ratio of these two: between-group variance / within-group variance, which is why the manually calculated ratio above matches the F-statistic printed in Task 2. A large F-statistic (large between-group variance relative to within-group variance) is what leads to a small p-value and a significant result.

task 5

Imagine you are analyzing Zomato order data for three different customer segments: Students, Working Professionals, and Retirees. Use ChatGPT or Copilot to generate Python code that performs a one-way ANOVA on their average monthly order amounts. Paste the generated code and run it with dummy data for each group.

In [5]:
# AI-generated code (from ChatGPT) for a one-way ANOVA on Zomato monthly order amounts
import numpy as np
from scipy import stats

np.random.seed(42)

# Dummy monthly order amount data (in rupees) for three Zomato customer segments
students = np.random.normal(loc=1800, scale=300, size=10)
working_professionals = np.random.normal(loc=3200, scale=400, size=10)
retirees = np.random.normal(loc=2200, scale=350, size=10)

print(f"Mean monthly order amount - Students: {round(np.mean(students), 2)}")
print(f"Mean monthly order amount - Working Professionals: {round(np.mean(working_professionals), 2)}")
print(f"Mean monthly order amount - Retirees: {round(np.mean(retirees), 2)}")

f_stat, p_value = stats.f_oneway(students, working_professionals, retirees)

print(f"\nF-statistic = {round(f_stat, 4)}")
print(f"p-value = {p_value:.6f}")

alpha = 0.05
if p_value < alpha:
    conclusion = "Reject H0 -> average monthly order amounts differ significantly across customer segments"
else:
    conclusion = "Fail to reject H0 -> no significant difference in average monthly order amounts"

print(f"Conclusion: {conclusion}")


Mean monthly order amount - Students: 1934.42
Mean monthly order amount - Working Professionals: 2883.74
Mean monthly order amount - Retirees: 2122.35

F-statistic = 34.5419
p-value = 0.000000
Conclusion: Reject H0 -> average monthly order amounts differ significantly across customer segments


**Explanation:** This follows the same one-way ANOVA approach as Task 2, but applied to a new scenario: dummy monthly order amounts for Students, Working Professionals, and Retirees on Zomato. Since Working Professionals were deliberately simulated with a much higher mean order amount (₹3200) than Students (₹1800) and Retirees (₹2200), the F-statistic comes out large and the p-value very small, so the ANOVA correctly detects that average monthly order amounts differ significantly across these three customer segments.